### **Prepare Train/Valve/Test Data**

##### **Overview**

Do not use random `train_test_split`: The model can "see the future" $\to$ Produce virtual results.

Chronological split 70/10/20

Why is separate VALIDATION necessary?

- We need to test `n_states=2` vs `n_states=3` vs `n_states=4`

- We must use an "unseen" dataset to choose the best one

- If we use tests to choose → data leakage → fictitious results (unintentional cheating)

- Rule: Test set can only be opened once when reporting the final results


**Import**

In [1]:
import sys
import os
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

**Set the project path**

In [2]:
base_dir = os.path.dirname(os.path.dirname(os.getcwd()))
sys.path.append(base_dir)
print(f"base_dir = {base_dir}")

data_dir = os.path.join(base_dir, 'data', 'processed_data')
print(f"data_dir = {data_dir}")

print(f"The files in data_dir: {os.listdir(data_dir)}")

base_dir = d:\Document\2025.2\IT3191E_Machine_Learning\ML_FinalProject_NILM_REFIT
data_dir = d:\Document\2025.2\IT3191E_Machine_Learning\ML_FinalProject_NILM_REFIT\data\processed_data
The files in data_dir: ['House2_full.csv', 'House2_part1.csv', 'House2_part2.csv', 'House2_part3.csv', 'House2_part4.csv', 'House2_part5.csv']


**Combine 5 CSV files into one `DataFrame`**

In [3]:
print("Reading data...")
dfs = []
for i in range(1, 6):  # i = 1, 2, 3, 4, 5
    filepath = os.path.join(data_dir, f'House2_part{i}.csv')
    print(f"  Reading House2_part{i}.csv...")
    df_part = pd.read_csv(filepath)
    dfs.append(df_part)

df = pd.concat(dfs, axis=0, ignore_index=True)
print(f"Number of rows: {len(df):,}")
print(f"Number of columns: {len(df.columns)}")
print(f"Column names: {df.columns.tolist()}")

Reading data...
  Reading House2_part1.csv...
  Reading House2_part2.csv...
  Reading House2_part3.csv...
  Reading House2_part4.csv...
  Reading House2_part5.csv...
Number of rows: 5,733,526
Number of columns: 12
Column names: ['Time', 'Unix', 'Aggregate', 'Appliance1', 'Appliance2', 'Appliance3', 'Appliance4', 'Appliance5', 'Appliance6', 'Appliance7', 'Appliance8', 'Appliance9']


**Save the merged df to a `.csv` file**

In [4]:
full_path = os.path.join(data_dir, 'House2_full.csv')

if not os.path.exists(full_path):
    print("Saving House2_full.csv...")
    df.to_csv(full_path, index=False)
    print(f"Saved at: {full_path}")
else:
    print(f"The file already exists")
    print("Reloading from House2_full.csv...")
    df = pd.read_csv(full_path)
    print(f"Loaded: {len(df):,} rows")


The file already exists
Reloading from House2_full.csv...
Loaded: 5,733,526 rows


**Process the `Time` column**

In [5]:
df['Time'] = pd.to_datetime(df['Time'])

print(f"Time data type: {df['Time'].dtype}") 
print(f"Start time: {df['Time'].min()}")
print(f"End time: {df['Time'].max()}")
print(f"Time interval: {df['Time'].max() - df['Time'].min()}")

time_diff = df['Time'].diff().dropna()
print(f"Mean distance between samples: {time_diff.mean()}")
print(f"Most common distance: {time_diff.mode()[0]}")

Time data type: datetime64[us]
Start time: 2013-09-17 22:08:11
End time: 2015-05-28 08:05:43
Time interval: 617 days 09:57:32
Mean distance between samples: 0 days 00:00:09.303988
Most common distance: 0 days 00:00:07


**Calculate `Appliance_Others`**

In [6]:
appliance_cols = ['Appliance1', 'Appliance2', 'Appliance3', 
                  'Appliance4', 'Appliance5', 'Appliance6',
                  'Appliance7', 'Appliance8', 'Appliance9']

df['sum_9_appliances'] = df[appliance_cols].sum(axis=1)

df['Appliance_Others'] = df['Aggregate'] - df['sum_9_appliances']

negative_count = (df['Appliance_Others'] < 0).sum()
print(f"Number of rows with Appliance_Others < 0 (to be deleted): {negative_count:,}")

df = df[df['Appliance_Others'] >= 0].copy()
print(f"The {negative_count:,} line has been removed. The current minimum value is: {df['Appliance_Others'].min()}")

df.drop(columns=['sum_9_appliances'], inplace=True)

print(f"Appliance_Others summarizes:")
print(df['Appliance_Others'].describe())

Number of rows with Appliance_Others < 0 (to be deleted): 28,444
The 28,444 line has been removed. The current minimum value is: 0
Appliance_Others summarizes:
count    5.705082e+06
mean     3.214029e+02
std      9.227989e+02
min      0.000000e+00
25%      8.300000e+01
50%      1.120000e+02
75%      2.300000e+02
max      2.438700e+04
Name: Appliance_Others, dtype: float64


**Add time features to the data.**

Why are time features needed?

- Washing machines often run between 8-10 AM or 7-9 PM.

- Microwaves are often used during mealtimes (12 PM, 6 PM).

- When the model knows "it's 3 AM," it will predict fewer appliances will be turned on.

In [7]:
df['hour'] = df['Time'].dt.hour
df['minute'] = df['Time'].dt.minute
df['dayofweek'] = df['Time'].dt.dayofweek
df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)

print("Time features added")
print(df[['Time', 'hour', 'minute', 'dayofweek', 'is_weekend']].head(3))

Time features added
                 Time  hour  minute  dayofweek  is_weekend
0 2013-09-17 22:08:11    22       8          1           0
1 2013-09-17 22:08:18    22       8          1           0
2 2013-09-17 22:08:26    22       8          1           0


**Divide train/val/test in chronological order (70/10/20)**

In [8]:
df = df.sort_values('Time').reset_index(drop=True)

total_rows = len(df)
train_end  = int(total_rows * 0.70)
val_end    = int(total_rows * 0.80)

df_train = df.iloc[:train_end].copy()
df_val   = df.iloc[train_end:val_end].copy()
df_test  = df.iloc[val_end:].copy()  

print(f"Total rows: {total_rows:,}")
print(f"Train: {len(df_train):,} rows ({len(df_train)/total_rows*100:.1f}%)")
print(f"Val: {len(df_val):,} rows ({len(df_val)/total_rows*100:.1f}%)")
print(f"Test: {len(df_test):,} rows ({len(df_test)/total_rows*100:.1f}%)")
print(f"\nTrain: {df_train['Time'].min()} → {df_train['Time'].max()}")
print(f"Val: {df_val['Time'].min()} → {df_val['Time'].max()}")
print(f"Test: {df_test['Time'].min()} → {df_test['Time'].max()}")

assert df_train['Time'].max() < df_val['Time'].min()
assert df_val['Time'].max()   < df_test['Time'].min()
print("\nNo overlap between sets")

Total rows: 5,705,082
Train: 3,993,557 rows (70.0%)
Val: 570,508 rows (10.0%)
Test: 1,141,017 rows (20.0%)

Train: 2013-09-17 22:08:11 → 2014-12-27 07:02:28
Val: 2014-12-27 07:02:35 → 2015-02-10 05:23:26
Test: 2015-02-10 05:23:33 → 2015-05-28 08:05:43

No overlap between sets


**Apply Window Shift to Train, Val, and Test**

In [9]:
from src.tools.window_shifter import WindowShifter

WINDOW_SIZE = 5

print(f"Applying WindowShifter with n={WINDOW_SIZE}...")
print("With the train set...")
df_train_shifted = WindowShifter.shift(df_train, n=WINDOW_SIZE)

print("With the validation set...")
df_val_shifted = WindowShifter.shift(df_val, n=WINDOW_SIZE)

print("With the test set...")
df_test_shifted = WindowShifter.shift(df_test, n=WINDOW_SIZE)

print(f"\nTrain shifted shape: {df_train_shifted.shape}")
print(f"Val shifted shape:   {df_val_shifted.shape}")
print(f"Test shifted shape:  {df_test_shifted.shape}")

print(df_train_shifted.head())

Applying WindowShifter with n=5...
With the train set...
With the validation set...
With the test set...

Train shifted shape: (3993553, 24)
Val shifted shape:   (570504, 24)
Test shifted shape:  (1141013, 24)
              Time_t0  Aggregate_t0             Time_t1  Aggregate_t1             Time_t2  Aggregate_t2             Time_t3  Aggregate_t3             Time_t4  Aggregate_t4  Appliance1  Appliance2  \
4 2013-09-17 22:08:42           700 2013-09-17 22:08:34         702.0 2013-09-17 22:08:26         694.0 2013-09-17 22:08:18         694.0 2013-09-17 22:08:11         695.0          88           0   
5 2013-09-17 22:08:50           702 2013-09-17 22:08:42         700.0 2013-09-17 22:08:34         702.0 2013-09-17 22:08:26         694.0 2013-09-17 22:08:18         694.0          88           0   
6 2013-09-17 22:08:58           696 2013-09-17 22:08:50         702.0 2013-09-17 22:08:42         700.0 2013-09-17 22:08:34         702.0 2013-09-17 22:08:26         694.0          88          

**Separate $X$ (features) and $y$ (targets) for train/val/test**

After this data splitting step:

|Variable|Function|
|-|-|
|`y_train`|`fit()` - train model|
|`agg_val`, `y_val`|grid search — select `n_states`/`n_iter`|
|`agg_test`, `y_test`|Final evaluation — OPENED ONLY ONCE|

In [10]:
target_cols = ['Appliance1', 'Appliance2', 'Appliance3',
               'Appliance4', 'Appliance5', 'Appliance6',
               'Appliance7', 'Appliance8', 'Appliance9',
               'Appliance_Others']

aggregate_feature_cols = [f'Aggregate_t{i}' for i in range(WINDOW_SIZE)]

X_train = df_train_shifted[aggregate_feature_cols].values
y_train = df_train_shifted[target_cols]

X_val   = df_val_shifted[aggregate_feature_cols].values
y_val   = df_val_shifted[target_cols]
agg_val = X_val[:, 0]

X_test  = df_test_shifted[aggregate_feature_cols].values
y_test  = df_test_shifted[target_cols]
agg_test = X_test[:, 0]

print("Sizes of the sets:")
print(f"  X_train: {X_train.shape}   y_train: {y_train.shape}")
print(f"  X_val:   {X_val.shape}     y_val:   {y_val.shape}")
print(f"  X_test:  {X_test.shape}    y_test:  {y_test.shape}")
print(f"\nEach row X = [{', '.join(aggregate_feature_cols[:3])}, ...]")
print(f"Example 3 top row X_train:")
print(X_train[:3, :])

Sizes of the sets:
  X_train: (3993553, 5)   y_train: (3993553, 10)
  X_val:   (570504, 5)     y_val:   (570504, 10)
  X_test:  (1141013, 5)    y_test:  (1141013, 10)

Each row X = [Aggregate_t0, Aggregate_t1, Aggregate_t2, ...]
Example 3 top row X_train:
[[700. 702. 694. 694. 695.]
 [702. 700. 702. 694. 694.]
 [696. 702. 700. 702. 694.]]
